## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-24-12-50-46 +0000,nyt,Stocks Set to Slide as Tech Stocks Partly Reve...,https://www.nytimes.com/2026/11/24/business/st...
1,2026-11-24-12-45-43 +0000,bbc,Russian governor detained in rare criminal cas...,https://www.bbc.co.uk/news/articles/cmwy868v3n...
2,2026-11-24-12-41-47 +0000,bbc,Prince of Wales makes first ever Downing Stree...,https://www.bbc.co.uk/news/videos/c6evr4vzydpy...
3,2026-11-24-12-36-00 +0000,wsj,The Morning Risk Report: U.S. Chip Restriction...,https://www.wsj.com/risk-compliance-journal/th...
4,2026-11-24-12-36-00 +0000,wsj,Burlington Stores Cuts Outlook as Quarterly Sa...,https://www.wsj.com/business/retail/burlington...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2704/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
197,huang,62
5,nvidia,54
196,jensen,48
195,trump,47
230,return,24


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
118,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...,289
357,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,253
290,2026-11-23-19-46-00 +0000,wsj,When President Trump wanted Jensen Huang back ...,https://www.wsj.com/politics/policy/nvidia-ceo...,252
68,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,252
340,2026-11-23-17-17-17 +0000,nypost,Trump posts edited image of assassination atte...,https://nypost.com/2026/11/23/us-news/trump-po...,250


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
118,289,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...
282,55,2026-11-23-20-13-46 +0000,nypost,Eric Swalwell’s pal Jimmy Gomez violated House...,https://nypost.com/2026/11/23/us-news/eric-swa...
153,54,2026-11-24-03-30-00 +0000,nypost,China’s second-richest man’s deal to drain Cal...,https://nypost.com/2026/11/23/us-news/chinese-...
259,51,2026-11-23-21-24-51 +0000,nypost,Nasdaq surges 6.5% in second-largest one-day g...,https://nypost.com/2026/11/23/business/nasdaq-...
146,49,2026-11-24-04-15-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
290,41,2026-11-23-19-46-00 +0000,wsj,When President Trump wanted Jensen Huang back ...,https://www.wsj.com/politics/policy/nvidia-ceo...
123,40,2026-11-24-07-47-34 +0000,nypost,Russia warns of ‘preventive nuclear attack’ as...,https://nypost.com/2026/11/24/world-news/russi...
95,39,2026-11-24-10-00-00 +0000,wsj,"In the next decade, America’s graying populati...",https://www.wsj.com/economy/home-health-aide-a...
275,38,2026-11-23-20-48-00 +0000,wsj,"Bayer Clears Major Roundup Hurdle, but a New S...",https://www.wsj.com/us-news/law/bayer-clears-m...
152,37,2026-11-24-03-38-00 +0000,wsj,Ilhan Omar Investigation Covers Most of Her Li...,https://www.wsj.com/politics/policy/ilhan-omar...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
